In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

log = pd.read_csv("log.csv")
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Helper to produce sorted unique list limited to max_items
def limited_sorted_unique(series, max_items=50):
    vals = series.dropna().unique()
    # Convert to strings for consistent sorting display if mixed types
    vals = [v for v in vals]
    try:
        sorted_vals = sorted(vals)
    except Exception:
        # fallback to string-sorted if direct sort fails
        sorted_vals = sorted(map(str, vals))
    return sorted_vals[:max_items]

# Summaries for each file
metric_rows = metric.shape[0]
metric_cmdbs = limited_sorted_unique(metric['cmdb_id'])
metric_kpis = limited_sorted_unique(metric['kpi_name'])

trace_rows = trace.shape[0]
trace_cmdbs = limited_sorted_unique(trace['cmdb_id'])
trace_kpis = limited_sorted_unique(trace['trace_name'])

log_rows = log.shape[0]
log_cmdbs = limited_sorted_unique(log['cmdb_id'])
log_kpis = limited_sorted_unique(log['log_name'])

error_rows = error_logs.shape[0]
error_cmdbs = limited_sorted_unique(error_logs['cmdb_id'])

# Assemble a compact pandas summary table
summary_df = pd.DataFrame([
    {
        "file": "metric.csv",
        "rows": metric_rows,
        "cmdb_ids": metric_cmdbs,
        "kpi_names": metric_kpis
    },
    {
        "file": "trace.csv",
        "rows": trace_rows,
        "cmdb_ids": trace_cmdbs,
        "kpi_names": trace_kpis
    },
    {
        "file": "log.csv",
        "rows": log_rows,
        "cmdb_ids": log_cmdbs,
        "kpi_names": log_kpis
    },
    {
        "file": "error_logs.csv",
        "rows": error_rows,
        "cmdb_ids": error_cmdbs,
        "kpi_names": None
    }
])

# Keep output compact
summary_df

```
Out[1]:
```
summary = (
    "Summary of loaded telemetry files:\n\n"
    f"metric.csv: {int(summary_df.loc[summary_df['file']=='metric.csv','rows'].iloc[0])} rows. "
    f"cmdb_ids: {', '.join(summary_df.loc[summary_df['file']=='metric.csv','cmdb_ids'].iloc[0])}. "
    f"Example kpi_names: {', '.join(summary_df.loc[summary_df['file']=='metric.csv','kpi_names'].iloc[0][:10])}.\n\n"
    f"trace.csv: {int(summary_df.loc[summary_df['file']=='trace.csv','rows'].iloc[0])} rows. "
    f"cmdb_ids: {', '.join(summary_df.loc[summary_df['file']=='trace.csv','cmdb_ids'].iloc[0])}. "
    f"Example trace_names: {', '.join(summary_df.loc[summary_df['file']=='trace.csv','kpi_names'].iloc[0][:10])}.\n\n"
    f"log.csv: {int(summary_df.loc[summary_df['file']=='log.csv','rows'].iloc[0])} rows. "
    f"cmdb_ids: {', '.join(summary_df.loc[summary_df['file']=='log.csv','cmdb_ids'].iloc[0])}. "
    f"log_name values: {', '.join(summary_df.loc[summary_df['file']=='log.csv','kpi_names'].iloc[0])}.\n\n"
    f"error_logs.csv: {int(summary_df.loc[summary_df['file']=='error_logs.csv','rows'].iloc[0])} rows. "
    f"cmdb_ids: {', '.join(summary_df.loc[summary_df['file']=='error_logs.csv','cmdb_ids'].iloc[0]) if summary_df.loc[summary_df['file']=='error_logs.csv','cmdb_ids'].iloc[0] else '[]'}.\n\n"
    "No thresholds or anomaly analysis performed yet."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

file  rows                                           cmdb_ids                                          kpi_names
0      metric.csv  1775  [adservice, cartservice, checkoutservice, curr...  [cpu, diskio, error, latency-50, latency-90, m...
1       trace.csv  3800  [checkoutservice, currencyservice, emailservic...  [trace.from_checkoutservice.duration_mean, tra...
2         log.csv   460  [adservice, cartservice, checkoutservice, curr...                   [log.error_count, log.row_count]
3  error_logs.csv     0                                                 []                                               None```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric DataFrame if present; otherwise reload
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# Ensure timestamp parsed as UTC
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate components
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Filter to candidate components
metric_cands = metric[metric['cmdb_id'].isin(candidates)].copy()

# Incident window (UTC), inclusive
window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
window_end = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")

# Compute global statistics per (cmdb_id, kpi_name)
group_all = (
    metric_cands
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count_all='count',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        min='min',
        max='max'
    )
    .reset_index()
)

# Compute counts within the incident window
mask = (metric_cands['timestamp'] >= window_start) & (metric_cands['timestamp'] <= window_end)
group_window = (
    metric_cands[mask]
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .count()
    .reset_index(name='count_in_window')
)

# Merge and fill missing window counts with 0
result_df = group_all.merge(group_window, on=['cmdb_id', 'kpi_name'], how='left')
result_df['count_in_window'] = result_df['count_in_window'].fillna(0).astype(int)

# Sort by count_in_window desc then cmdb_id and limit to top 100
result_df = result_df.sort_values(by=['count_in_window', 'cmdb_id'], ascending=[False, True])

# Keep columns in required order and limit rows
result_df = result_df[['cmdb_id', 'kpi_name', 'count_all', 'p95', 'p5', 'min', 'max', 'count_in_window']].head(100)

# Display compact result
result_df

```
Out[2]:
```
summary = (
    f"Telemetry summary for candidate components (metric.csv):\n\n"
    f"- Analyzed (cmdb_id, kpi_name) pairs: {result_df.shape[0]}.\n"
    f"- Components present: {', '.join(sorted(result_df['cmdb_id'].unique()))}.\n"
    f"- All pairs have the same count_in_window: {result_df['count_in_window'].unique().tolist()} (samples inside 2024-01-15 17:27:22 to 2024-01-15 17:57:22 UTC).\n"
    f"- Example KPI names (up to 50): {', '.join(sorted(result_df['kpi_name'].unique())[:50])}.\n\n"
    "Interpretation: For the candidate services, each (service, KPI) time series has full coverage inside the specified incident window (non-zero and equal counts). Global statistics (count_all, p95, p5, min, max) were computed per pair and are available in the table."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_all           p95            p5           min           max  count_in_window
0         checkoutservice         cpu         25  4.739594e-01  2.956188e-01  2.164325e-01  5.118590e-01               25
1         checkoutservice  latency-50         25  4.193414e-01  2.097742e-01  8.378759e-02  5.021726e-01               25
2         checkoutservice  latency-90         25  1.252759e+00  5.068220e-01  4.735961e-01  1.368458e+00               25
3         checkoutservice         mem         25  1.167566e+07  1.106728e+07  1.097810e+07  1.184597e+07               25
4         checkoutservice      socket         25  9.000000e+00  9.000000e+00  9.000000e+00  9.000000e+00               25
..                    ...         ...        ...           ...           ...           ...           ...              ...
27  recommendationservice  latency-50         25  3.113243e-02  6.994136e-03  6.659449e-03  3.161526e-02               25
28  recommendationservice  latency-90         25  6.382920e-02  9.442345e-03  9.420788e-03  6.618097e-02               25
29  recommendationservice         mem         25  4.681981e+07  4.630752e+07  4.627483e+07  4.686679e+07               25
30  recommendationservice      socket         25  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00               25
31  recommendationservice    workload         25  1.883672e+01  1.691175e+01  1.492105e+01  1.931902e+01               25

[32 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric if already loaded; otherwise load and parse timestamp as UTC
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed correctly (idempotent)
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate components
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Filter to candidate services
metric_cands = metric[metric['cmdb_id'].isin(candidates)].copy()

# 1) Compute global p95 and p5 per (cmdb_id, kpi_name) using full series (global thresholds)
thresholds = (
    metric_cands
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter rows to incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
window_end = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")
mask = (metric_cands['timestamp'] >= window_start) & (metric_cands['timestamp'] <= window_end)
window_df = metric_cands[mask].copy()

# Merge thresholds into windowed data
window_df = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies: high if value >= p95, low if value <= p5
window_df['is_high'] = window_df['value'] >= window_df['p95']
window_df['is_low'] = window_df['value'] <= window_df['p5']
window_df['is_anom'] = window_df['is_high'] | window_df['is_low']

# Compute relative violation per row for anomalies:
# For high: (value - p95)/p95 ; for low: (p5 - value)/p5. Handle zero p95/p5 by using NaN.
window_df['rel_high'] = np.where(window_df['p95'] != 0, (window_df['value'] - window_df['p95']) / window_df['p95'], np.nan)
window_df['rel_low']  = np.where(window_df['p5']  != 0, (window_df['p5'] - window_df['value']) / window_df['p5'], np.nan)
# Use absolute positive magnitudes; set rel_violation to appropriate value per row if anomaly, otherwise NaN
window_df['rel_violation'] = np.where(window_df['is_high'], window_df['rel_high'],
                                      np.where(window_df['is_low'], window_df['rel_low'], np.nan))
# Ensure non-negative
window_df['rel_violation'] = window_df['rel_violation'].abs()

# Aggregate per (cmdb_id, kpi_name)
agg = (
    window_df
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        total_points_in_window=('value', 'count'),
        anomaly_count_in_window=('is_anom', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[window_df.loc[x.index, 'is_anom']].min() if any(window_df.loc[x.index, 'is_anom']) else pd.NaT),
        count_high=('is_high', 'sum'),
        count_low=('is_low', 'sum'),
        max_rel_violation=('rel_violation', 'max')
    )
    .reset_index()
)

# Determine anomaly_direction per group: prefer direction with more anomalies; tie -> 'high'
def choose_direction(row):
    if row['count_high'] >= row['count_low'] and row['count_high'] > 0:
        return 'high'
    elif row['count_low'] >  row['count_high']:
        return 'low'
    else:
        return None

agg['anomaly_direction'] = agg.apply(choose_direction, axis=1)

# Keep only groups with anomaly_count_in_window > 0
agg = agg[agg['anomaly_count_in_window'] > 0].copy()

# Select and rename columns as requested
result = agg[['cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count_in_window',
              'earliest_anomaly_timestamp', 'anomaly_direction', 'max_rel_violation']].copy()

# Sort by anomaly_count_in_window desc, then max_relative_violation desc
result = result.sort_values(by=['anomaly_count_in_window', 'max_rel_violation'], ascending=[False, False]).head(20)

# Format earliest timestamp as UTC (keep as pd.Timestamp)
# Prepare example anomaly rows: up to 3 example anomalies per top 5 pairs
top5_pairs = result[['cmdb_id', 'kpi_name']].head(5)
examples_list = []
for _, r in top5_pairs.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    sel = window_df[
        (window_df['cmdb_id'] == cmdb) &
        (window_df['kpi_name'] == kpi) &
        (window_df['is_anom'])
    ].sort_values('timestamp').head(3)
    if not sel.empty:
        examples_list.append(sel[['timestamp', 'cmdb_id', 'kpi_name', 'value']])

if examples_list:
    examples_df = pd.concat(examples_list, ignore_index=True)
else:
    examples_df = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'kpi_name', 'value'])

# Keep outputs compact: result (top up to 20 rows) and examples_df (<= 15 rows)
result, examples_df

```
Out[3]:
```
summary = (
    "Summary of anomaly scan for candidate services (window 2024-01-15 17:27:22 to 2024-01-15 17:57:22 UTC):\n\n"
    "- The dominant and most widespread anomalies are in the 'socket' KPI across services. "
    "checkoutservice, currencyservice, emailservice, and recommendationservice each had 25/25 points flagged as anomalies for 'socket'; "
    "productcatalogservice had 24/25. Earliest socket anomalies appear at 2024-01-15 17:30:00 UTC.\n\n"
    "- diskio for emailservice and recommendationservice also showed all-window anomalies (25/25) but relative-violation values are NaN (likely due to a zero threshold or division-by-zero in relative calculation).\n\n"
    "- emailservice shows many low-end latency anomalies: latency-50 and latency-90 each had 18 anomalous points (direction = low) with small relative deviations.\n\n"
    "- Several services/KPIs (cpu, workload, latency) have smaller numbers of high anomalies (typically 4 points) with modest relative violations (examples include checkoutservice latency-50 and productcatalogservice cpu).\n\n"
    "Interpretation: The clearest signal is a socket-related issue affecting most candidate services starting about 17:30 UTC. Other issues (diskio zeros and some latency degradations on emailservice) are present but less widespread.\n"
)
summary, examples_df.head(15)

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp anomaly_direction  max_rel_violation
4         checkoutservice      socket                      25                       25  2024-01-15 17:30:00+00:00              high           0.000000
10        currencyservice      socket                      25                       25  2024-01-15 17:30:00+00:00              high           0.000000
17           emailservice      socket                      25                       25  2024-01-15 17:30:00+00:00              high           0.000000
30  recommendationservice      socket                      25                       25  2024-01-15 17:30:00+00:00              high           0.000000
13           emailservice      diskio                      25                       25  2024-01-15 17:30:00+00:00              high                NaN
26  recommendationservice      diskio                      25                       25  2024-01-15 17:30:00+00:00              high                NaN
23  productcatalogservice      socket                      25                       24  2024-01-15 17:30:00+00:00              high           0.000000
15           emailservice  latency-90                      25                       18  2024-01-15 17:30:00+00:00               low           0.002697
14           emailservice  latency-50                      25                       18  2024-01-15 17:30:00+00:00               low           0.002306
1         checkoutservice  latency-50                      25                        4  2024-01-15 17:30:00+00:00              high           0.600582
19  productcatalogservice         cpu                      25                        4  2024-01-15 17:30:00+00:00              high           0.413438
18           emailservice    workload                      25                        4  2024-01-15 17:30:00+00:00              high           0.370350
0         checkoutservice         cpu                      25                        4  2024-01-15 17:30:00+00:00              high           0.267866
24  productcatalogservice    workload                      25                        4  2024-01-15 17:30:00+00:00              high           0.222820
5         checkoutservice    workload                      25                        4  2024-01-15 17:30:00+00:00              high           0.131304
31  recommendationservice    workload                      25                        4  2024-01-15 17:30:00+00:00              high           0.117711
2         checkoutservice  latency-90                      25                        4  2024-01-15 17:31:00+00:00              high           0.092355
25  recommendationservice         cpu                      25                        4  2024-01-15 17:30:00+00:00              high           0.077437
6         currencyservice         cpu                      25                        4  2024-01-15 17:30:00+00:00              high           0.057681
27  recommendationservice  latency-50                      25                        4  2024-01-15 17:30:00+00:00              high           0.047852,                    timestamp                cmdb_id kpi_name  value
0  2024-01-15 17:30:00+00:00        checkoutservice   socket    9.0
1  2024-01-15 17:31:00+00:00        checkoutservice   socket    9.0
2  2024-01-15 17:32:00+00:00        checkoutservice   socket    9.0
3  2024-01-15 17:30:00+00:00        currencyservice   socket    3.0
4  2024-01-15 17:31:00+00:00        currencyservice   socket    3.0
5  2024-01-15 17:32:00+00:00        currencyservice   socket    3.0
6  2024-01-15 17:30:00+00:00           emailservice   socket    3.0
7  2024-01-15 17:31:00+00:00           emailservice   socket    3.0
8  2024-01-15 17:32:00+00:00           emailservice   socket    3.0
9  2024-01-15 17:30:00+00:00  recommendationservice   socket    4.0
10 2024-01-15 17:31:00+00:00  recommendationservice   socket    4.0
11 2024-01-15 17:32:00+00:00  recommendationservice   socket    4.0
12 2024-01-15 17:30:00+00:00           emailservice   diskio    0.0
13 2024-01-15 17:31:00+00:00           emailservice   diskio    0.0
14 2024-01-15 17:32:00+00:00           emailservice   diskio    0.0)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing trace DataFrame if present; otherwise load and parse timestamp as UTC
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed correctly (idempotent)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Candidate components
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Restrict to candidate services
trace_cands = trace[trace['cmdb_id'].isin(candidates)].copy()

# Compute global p95 per (cmdb_id, trace_name) using the full series (before filtering)
trace_thresholds = (
    trace_cands
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
window_end = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")

# Filter rows to the incident window
mask = (trace_cands['timestamp'] >= window_start) & (trace_cands['timestamp'] <= window_end)
trace_window = trace_cands[mask].copy()

# Merge thresholds into windowed data
trace_window = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Identify anomalies: value >= p95
trace_window['is_anom'] = trace_window['value'] >= trace_window['p95']

# Compute relative violation per anomaly: (value - p95)/p95, handle p95==0 by producing NaN
trace_window['rel_violation'] = np.where(
    (trace_window['is_anom']) & (trace_window['p95'] != 0),
    (trace_window['value'] - trace_window['p95']) / trace_window['p95'],
    np.nan
)
# Keep only anomalies where is_anom True for some aggregations
# Aggregate per (cmdb_id, trace_name)
agg = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'])
    .agg(
        total_points_in_window=('value', 'count'),
        anomaly_count_in_window=('is_anom', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[trace_window.loc[x.index, 'is_anom']].min() if any(trace_window.loc[x.index, 'is_anom']) else pd.NaT),
        max_relative_violation=('rel_violation', 'max')
    )
    .reset_index()
)

# Keep only rows with anomaly_count_in_window > 0
agg = agg[agg['anomaly_count_in_window'] > 0].copy()

# Sort by anomaly_count_in_window desc then max_relative_violation desc, limit to top 20
agg = agg.sort_values(by=['anomaly_count_in_window', 'max_relative_violation'], ascending=[False, False]).head(20)

# Prepare up to 5 example anomaly rows for top 3 pairs
top3 = agg[['cmdb_id', 'trace_name']].head(3)
examples_list = []
for _, row in top3.iterrows():
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    sel = trace_window[
        (trace_window['cmdb_id'] == cmdb) &
        (trace_window['trace_name'] == tname) &
        (trace_window['is_anom'])
    ].sort_values('timestamp').head(5)
    if not sel.empty:
        examples_list.append(sel[['timestamp', 'cmdb_id', 'trace_name', 'value']])

if examples_list:
    trace_examples = pd.concat(examples_list, ignore_index=True)
else:
    trace_examples = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'trace_name', 'value'])

# Keep outputs compact: agg (top <=20 rows) and trace_examples (<=15 rows)
agg, trace_examples

```
Out[4]:
```
summary = (
    "Trace anomaly summary (window 2024-01-15 17:27:22 to 2024-01-15 17:57:22 UTC):\n\n"
    "- Many trace error_rate metrics across the candidate services were flagged as anomalies. "
    "Numerous (cmdb_id, trace_name) pairs (e.g., multiple trace.from_*.error_rate and trace.to_*.error_rate for checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice) had anomaly_count_in_window = 25 (i.e., every sampled point in the 30‑minute window was marked anomalous).\n\n"
    "- The earliest flagged anomalies appear at 2024-01-15 17:30:00 UTC for these pairs.\n\n"
    "- The reported max_relative_violation values are NaN for the top pairs: this indicates the global p95 threshold is zero (or effectively zero), so relative violation cannot be computed. The example trace rows show values = 0.0, which meet the condition value >= p95 when p95 == 0, causing every point to be labeled an anomaly. This is likely a thresholding artifact rather than a clear spike in trace error rates.\n\n"
    "Conclusion: Many trace error_rate series are flagged because their global p95 is zero and current values equal or exceed that zero threshold. Investigate the thresholding (zero p95) and inspect raw trace semantics before treating these as true error spikes."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp  max_relative_violation
2         checkoutservice        trace.from_checkoutservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
6         checkoutservice        trace.from_frontendservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
10        checkoutservice                   trace.from_root.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
14        checkoutservice          trace.to_checkoutservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
18        checkoutservice          trace.to_currencyservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
22        checkoutservice             trace.to_emailservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
26        checkoutservice           trace.to_paymentservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
30        checkoutservice    trace.to_productcatalogservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
34        currencyservice        trace.from_checkoutservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
38        currencyservice        trace.from_frontendservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
42        currencyservice                   trace.from_root.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
46           emailservice        trace.from_checkoutservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
50           emailservice                   trace.from_root.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
54  productcatalogservice        trace.from_checkoutservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
58  productcatalogservice        trace.from_frontendservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
62  productcatalogservice  trace.from_recommendationservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
66  productcatalogservice                   trace.from_root.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
70  recommendationservice        trace.from_frontendservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
74  recommendationservice  trace.from_recommendationservice.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN
78  recommendationservice                   trace.from_root.error_rate                      25                       25  2024-01-15 17:30:00+00:00                     NaN,                    timestamp          cmdb_id                             trace_name  value
0  2024-01-15 17:30:00+00:00  checkoutservice  trace.from_checkoutservice.error_rate    0.0
1  2024-01-15 17:31:00+00:00  checkoutservice  trace.from_checkoutservice.error_rate    0.0
2  2024-01-15 17:32:00+00:00  checkoutservice  trace.from_checkoutservice.error_rate    0.0
3  2024-01-15 17:33:00+00:00  checkoutservice  trace.from_checkoutservice.error_rate    0.0
4  2024-01-15 17:34:00+00:00  checkoutservice  trace.from_checkoutservice.error_rate    0.0
5  2024-01-15 17:30:00+00:00  checkoutservice  trace.from_frontendservice.error_rate    0.0
6  2024-01-15 17:31:00+00:00  checkoutservice  trace.from_frontendservice.error_rate    0.0
7  2024-01-15 17:32:00+00:00  checkoutservice  trace.from_frontendservice.error_rate    0.0
8  2024-01-15 17:33:00+00:00  checkoutservice  trace.from_frontendservice.error_rate    0.0
9  2024-01-15 17:34:00+00:00  checkoutservice  trace.from_frontendservice.error_rate    0.0
10 2024-01-15 17:30:00+00:00  checkoutservice             trace.from_root.error_rate    0.0
11 2024-01-15 17:31:00+00:00  checkoutservice             trace.from_root.error_rate    0.0
12 2024-01-15 17:32:00+00:00  checkoutservice             trace.from_root.error_rate    0.0
13 2024-01-15 17:33:00+00:00  checkoutservice             trace.from_root.error_rate    0.0
14 2024-01-15 17:34:00+00:00  checkoutservice             trace.from_root.error_rate    0.0)```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse existing 'log' DataFrame if present; otherwise load and parse timestamp as UTC
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Candidate services (reuse from earlier if available)
try:
    candidates
except NameError:
    candidates = [
        "checkoutservice",
        "currencyservice",
        "emailservice",
        "productcatalogservice",
        "recommendationservice"
    ]

# Incident window (reuse if available)
try:
    window_start, window_end
except NameError:
    window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
    window_end   = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")

# Filter log rows to candidate services and incident window (inclusive)
log_window = log[
    (log['cmdb_id'].isin(candidates)) &
    (log['timestamp'] >= window_start) &
    (log['timestamp'] <= window_end)
].copy()

# Ensure numeric values
log_window['value'] = pd.to_numeric(log_window['value'], errors='coerce')

# total_rows_in_window per cmdb_id (count of log rows)
total_rows = log_window.groupby('cmdb_id').size().reset_index(name='total_rows_in_window')

# sum_log_error_count: sum of value where log_name == 'log.error_count'
err_sum = (
    log_window[log_window['log_name'] == 'log.error_count']
    .groupby('cmdb_id')['value']
    .sum()
    .reset_index(name='sum_log_error_count')
)

# mean_log_row_count: mean of value where log_name == 'log.row_count'
rowcount_mean = (
    log_window[log_window['log_name'] == 'log.row_count']
    .groupby('cmdb_id')['value']
    .mean()
    .reset_index(name='mean_log_row_count')
)

# Assemble results for all candidate services (ensure all candidates present)
result_df = pd.DataFrame({'cmdb_id': candidates}).merge(total_rows, on='cmdb_id', how='left')
result_df = result_df.merge(err_sum, on='cmdb_id', how='left')
result_df = result_df.merge(rowcount_mean, on='cmdb_id', how='left')

# Fill missing numeric values with 0 (no rows -> zeros)
result_df['total_rows_in_window'] = result_df['total_rows_in_window'].fillna(0).astype(int)
result_df['sum_log_error_count'] = result_df['sum_log_error_count'].fillna(0.0)
result_df['mean_log_row_count'] = result_df['mean_log_row_count'].fillna(0.0)

# Sort by sum_log_error_count desc
result_df = result_df[['cmdb_id', 'total_rows_in_window', 'sum_log_error_count', 'mean_log_row_count']]
result_df = result_df.sort_values(by='sum_log_error_count', ascending=False).reset_index(drop=True)

# Prepare example log rows:
# If any service has sum_log_error_count > 0, return up to 5 example rows for those services where log_name == 'log.error_count'
services_with_errors = result_df.loc[result_df['sum_log_error_count'] > 0, 'cmdb_id'].tolist()
if services_with_errors:
    examples_df = (
        log_window[
            (log_window['cmdb_id'].isin(services_with_errors)) &
            (log_window['log_name'] == 'log.error_count')
        ]
        .sort_values(['cmdb_id', 'timestamp'])
        .loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']]
        .head(5)
        .reset_index(drop=True)
    )
else:
    # No errors: return top 5 rows by log.row_count value within the window
    examples_df = (
        log_window[log_window['log_name'] == 'log.row_count']
        .sort_values('value', ascending=False)
        .loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']]
        .head(5)
        .reset_index(drop=True)
    )

# Keep outputs compact
result_df, examples_df

```
Out[5]:
```
summary = (
    "Log summary for candidate services in window 2024-01-15 17:27:22 to 2024-01-15 17:57:22 UTC:\n\n"
    "- All candidate services have no log.error_count during the window (sum_log_error_count = 0 for every service).\n\n"
    "- Per-service metrics:\n"
    "  * checkoutservice: total_rows_in_window = 50, mean log.row_count = 76.84\n"
    "  * currencyservice: total_rows_in_window = 50, mean log.row_count = 2226.56\n"
    "  * emailservice: total_rows_in_window = 50, mean log.row_count = 25.60\n"
    "  * productcatalogservice: total_rows_in_window = 0 (no log rows in window)\n"
    "  * recommendationservice: total_rows_in_window = 50, mean log.row_count = 515.24\n\n"
    "- Because no log.error_count > 0 was observed, there are no error examples to show. The top example log rows (by log.row_count) come from currencyservice, e.g. timestamps around 2024-01-15 17:31–17:37 UTC with row_count values like 2490, 2442, 2406.\n\n"
    "Conclusion: No log.error_count anomalies were observed for the candidate services in the incident window. Currencyservice shows substantially higher log row counts than other services during that period, while productcatalogservice had no log data in the window."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id  total_rows_in_window  sum_log_error_count  mean_log_row_count
0        checkoutservice                    50                  0.0               76.84
1        currencyservice                    50                  0.0             2226.56
2           emailservice                    50                  0.0               25.60
3  productcatalogservice                     0                  0.0                0.00
4  recommendationservice                    50                  0.0              515.24,                   timestamp          cmdb_id       log_name  value
0 2024-01-15 17:32:00+00:00  currencyservice  log.row_count   2490
1 2024-01-15 17:31:00+00:00  currencyservice  log.row_count   2490
2 2024-01-15 17:35:00+00:00  currencyservice  log.row_count   2442
3 2024-01-15 17:45:00+00:00  currencyservice  log.row_count   2406
4 2024-01-15 17:37:00+00:00  currencyservice  log.row_count   2404)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse existing metric if available; otherwise load and parse timestamp as UTC
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate services
try:
    candidates
except NameError:
    candidates = [
        "checkoutservice",
        "currencyservice",
        "emailservice",
        "productcatalogservice",
        "recommendationservice"
    ]

# Incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
window_end   = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")

# Restrict metric to candidates and kpi_name == 'socket' for processing
metric_socket = metric[(metric['cmdb_id'].isin(candidates)) & (metric['kpi_name'] == 'socket')].copy()

# 1) Compute global p95 and full-series count for each cmdb_id (do NOT filter by time)
global_stats = (
    metric_socket
    .groupby('cmdb_id')['value']
    .agg(count_all='count', p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Filter rows to the incident window (inclusive)
mask = (metric_socket['timestamp'] >= window_start) & (metric_socket['timestamp'] <= window_end)
socket_window = metric_socket[mask].copy()

# 3) For each cmdb_id compute window aggregates and anomaly metrics
# Merge global p95 into windowed rows for relative calculations
socket_window = socket_window.merge(global_stats[['cmdb_id', 'p95']], on='cmdb_id', how='left')

# Mark anomalies in window: value >= p95
socket_window['is_anom'] = socket_window['value'] >= socket_window['p95']

# Compute relative violation per anomaly: (value - p95)/p95 ; if p95 == 0 -> NaN
socket_window['rel_violation'] = np.where(socket_window['p95'] != 0,
                                         (socket_window['value'] - socket_window['p95']) / socket_window['p95'],
                                         np.nan)
# Only keep rel_violation for anomalies
socket_window.loc[~socket_window['is_anom'], 'rel_violation'] = np.nan
# Ensure absolute ratio (though for high anomalies it will be >=0)
socket_window['rel_violation'] = socket_window['rel_violation'].abs()

# Aggregate per cmdb_id
agg_window = (
    socket_window
    .groupby('cmdb_id')
    .agg(
        total_points_in_window=('value', 'count'),
        anomaly_count_in_window=('is_anom', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[socket_window.loc[x.index, 'is_anom']].min() if any(socket_window.loc[x.index, 'is_anom']) else pd.NaT),
        mean_value_in_window=('value', 'mean'),
        max_value_in_window=('value', 'max'),
        max_relative_violation_in_window=('rel_violation', 'max')
    )
    .reset_index()
)

# Combine with global_stats (to include count_all and p95). Ensure all candidate services present.
result_df = pd.DataFrame({'cmdb_id': candidates}).merge(global_stats, on='cmdb_id', how='left')
result_df = result_df.merge(agg_window, on='cmdb_id', how='left')

# Fill missing numeric window aggregates with 0 or NaN as appropriate
result_df['count_all'] = result_df['count_all'].fillna(0).astype(int)
result_df['p95'] = result_df['p95']  # keep NaN if missing
result_df['total_points_in_window'] = result_df['total_points_in_window'].fillna(0).astype(int)
result_df['anomaly_count_in_window'] = result_df['anomaly_count_in_window'].fillna(0).astype(int)
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_timestamp']  # NaT if none
result_df['mean_value_in_window'] = result_df['mean_value_in_window'].astype(float)
result_df['max_value_in_window'] = result_df['max_value_in_window'].astype(float)
result_df['max_relative_violation_in_window'] = result_df['max_relative_violation_in_window'].astype(float)

# Per requirement: if p95 == 0 set relative violation to NaN (already handled per-row; ensure group-level)
result_df.loc[result_df['p95'] == 0, 'max_relative_violation_in_window'] = np.nan

# Sort by anomaly_count_in_window desc then max_value_in_window desc
result_df = result_df.sort_values(by=['anomaly_count_in_window', 'max_value_in_window'], ascending=[False, False])

# Keep only requested columns and limit to candidate services (max 5 rows)
result_df = result_df[['cmdb_id', 'count_all', 'p95', 'total_points_in_window',
                       'anomaly_count_in_window', 'earliest_anomaly_timestamp',
                       'mean_value_in_window', 'max_value_in_window', 'max_relative_violation_in_window']]

# Prepare up to 10 example socket rows from the window ordered by timestamp asc
examples_df = socket_window.sort_values('timestamp').loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']].head(10).reset_index(drop=True)

# Display compact outputs
result_df, examples_df

```
Out[7]:
```
summary = (
    "Socket KPI summary for candidate services (window 2024-01-15 17:27:22 to 2024-01-15 17:57:22 UTC):\n\n"
    "- checkoutservice: count_all=25, p95=9.0, points_in_window=25, anomalies=25, earliest=2024-01-15 17:30:00 UTC, "
    "mean_in_window=9.0, max_in_window=9.0, max_relative_violation=0.0\n"
    "- recommendationservice: count_all=25, p95=4.0, points_in_window=25, anomalies=25, earliest=2024-01-15 17:30:00 UTC, "
    "mean_in_window=4.0, max_in_window=4.0, max_relative_violation=0.0\n"
    "- currencyservice: count_all=25, p95=3.0, points_in_window=25, anomalies=25, earliest=2024-01-15 17:30:00 UTC, "
    "mean_in_window=3.0, max_in_window=3.0, max_relative_violation=0.0\n"
    "- emailservice: count_all=25, p95=3.0, points_in_window=25, anomalies=25, earliest=2024-01-15 17:30:00 UTC, "
    "mean_in_window=3.0, max_in_window=3.0, max_relative_violation=0.0\n"
    "- productcatalogservice: count_all=25, p95=6.0, points_in_window=25, anomalies=12, earliest=2024-01-15 17:43:00 UTC, "
    "mean_in_window≈5.0053, max_in_window=6.0, max_relative_violation=0.0\n\n"
    "Interpretation: A socket-related signal affects all candidate services beginning around 17:30 UTC. For four services every sampled point in the window met or exceeded the global p95 (anomaly_count == points_in_window). The computed max_relative_violation is 0 for these series because the window values equal the p95 thresholds (i.e., values equal the p95, not larger), which may indicate a thresholding artifact rather than large spikes. Productcatalogservice shows fewer socket anomalies (12/25) with the first anomaly later at ~17:43 UTC.\n\n"
    "Examples from the window (timestamps show the start of anomalies): checkoutservice socket = 9.0 at 17:30 UTC; currencyservice socket = 3.0 at 17:30 UTC; emailservice socket = 3.0 at 17:30 UTC; recommendationservice socket = 4.0 at 17:30 UTC.\n\n"
    "Recommendation: investigate socket metrics and the global p95 thresholds (why p95 equals current values) and inspect socket-related logs/traces for root cause."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id  count_all  p95  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp  mean_value_in_window  max_value_in_window  max_relative_violation_in_window
0        checkoutservice         25  9.0                      25                       25  2024-01-15 17:30:00+00:00              9.000000                  9.0                               0.0
4  recommendationservice         25  4.0                      25                       25  2024-01-15 17:30:00+00:00              4.000000                  4.0                               0.0
1        currencyservice         25  3.0                      25                       25  2024-01-15 17:30:00+00:00              3.000000                  3.0                               0.0
2           emailservice         25  3.0                      25                       25  2024-01-15 17:30:00+00:00              3.000000                  3.0                               0.0
3  productcatalogservice         25  6.0                      25                       12  2024-01-15 17:43:00+00:00              5.005333                  6.0                               0.0,                   timestamp                cmdb_id kpi_name  value
0 2024-01-15 17:30:00+00:00        checkoutservice   socket    9.0
1 2024-01-15 17:30:00+00:00        currencyservice   socket    3.0
2 2024-01-15 17:30:00+00:00           emailservice   socket    3.0
3 2024-01-15 17:30:00+00:00  productcatalogservice   socket    4.0
4 2024-01-15 17:30:00+00:00  recommendationservice   socket    4.0
5 2024-01-15 17:31:00+00:00  recommendationservice   socket    4.0
6 2024-01-15 17:31:00+00:00           emailservice   socket    3.0
7 2024-01-15 17:31:00+00:00  productcatalogservice   socket    4.0
8 2024-01-15 17:31:00+00:00        checkoutservice   socket    9.0
9 2024-01-15 17:31:00+00:00        currencyservice   socket    3.0)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np

# Reuse existing trace DataFrame if present; otherwise load and parse timestamp as UTC
try:
    trace
except NameError:
    trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Candidate services
candidates = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Restrict to candidate services
trace_cands = trace[trace['cmdb_id'].isin(candidates)].copy()

# Identify trace_name values containing substring 'duration' (case-sensitive)
mask_duration = trace_cands['trace_name'].str.contains('duration', na=False)
trace_duration = trace_cands[mask_duration].copy()

# Compute global p95 per (cmdb_id, trace_name) using full series (before filtering)
duration_p95 = (
    trace_duration
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95))
    .reset_index()
)

# Incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
window_end   = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")

# Filter rows to the incident window
mask_win = (trace_duration['timestamp'] >= window_start) & (trace_duration['timestamp'] <= window_end)
trace_window = trace_duration[mask_win].copy()

# Merge global p95 into windowed data
trace_window = trace_window.merge(duration_p95, on=['cmdb_id', 'trace_name'], how='left')

# Identify anomalies: value >= p95
trace_window['is_anom'] = trace_window['value'] >= trace_window['p95']

# Compute relative violation per anomaly: (value - p95)/p95 ; if p95 == 0 -> NaN
trace_window['rel_violation'] = np.where(
    (trace_window['is_anom']) & (trace_window['p95'] != 0),
    (trace_window['value'] - trace_window['p95']) / trace_window['p95'],
    np.nan
)
# Keep absolute positive magnitudes
trace_window['rel_violation'] = trace_window['rel_violation'].abs()

# Aggregate per (cmdb_id, trace_name)
agg = trace_window.groupby(['cmdb_id', 'trace_name']).apply(
    lambda g: pd.Series({
        'p95': g['p95'].iloc[0],
        'total_points_in_window': int(len(g)),
        'anomaly_count_in_window': int(g['is_anom'].sum()),
        'earliest_anomaly_timestamp': g.loc[g['is_anom'], 'timestamp'].min() if g['is_anom'].any() else pd.NaT,
        'max_relative_violation': g.loc[g['is_anom'], 'rel_violation'].max() if g['is_anom'].any() else np.nan
    })
).reset_index()

# Keep only rows with anomaly_count_in_window > 0
agg = agg[agg['anomaly_count_in_window'] > 0].copy()

# Sort by anomaly_count_in_window desc then max_relative_violation desc, limit top 20
agg = agg.sort_values(by=['anomaly_count_in_window', 'max_relative_violation'], ascending=[False, False]).head(20)

# Prepare up to 5 example trace anomaly rows for the top 3 (cmdb_id, trace_name) pairs
top3 = agg[['cmdb_id', 'trace_name']].head(3)
examples_list = []
for _, row in top3.iterrows():
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    sel = trace_window[
        (trace_window['cmdb_id'] == cmdb) &
        (trace_window['trace_name'] == tname) &
        (trace_window['is_anom'])
    ].sort_values('timestamp').head(5)
    if not sel.empty:
        examples_list.append(sel[['timestamp', 'cmdb_id', 'trace_name', 'value']])

if examples_list:
    examples_df = pd.concat(examples_list, ignore_index=True)
else:
    examples_df = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'trace_name', 'value'])

# Keep outputs compact: agg (top <=20 rows) and examples_df (<=15 rows)
agg, examples_df

```
Out[8]:
```
summary = (
    "Trace-duration anomaly summary (window 2024-01-15 17:27:22 to 2024-01-15 17:57:22 UTC):\n\n"
    "- The most pronounced duration anomalies are associated with currencyservice (trace.from_root.duration_p95 and trace.from_root.duration_mean). "
    "These show very large relative violations (e.g. ~149x and ~82x the global p95) with earliest anomalies at 2024-01-15 17:36:00 UTC. Example spike: trace.from_root.duration_p95 = 0.038986 at 17:36 UTC.\n\n"
    "- Many other duration metrics across services (checkoutservice, productcatalogservice, recommendationservice, emailservice) each have 2 anomaly points in the window. Notable pairs include calls involving to_currencyservice, from_checkoutservice, to_paymentservice, and various from_*/to_* duration_mean and duration_p95 metrics.\n\n"
    "- Pattern: anomalies are sparse but impactful (typically 2 anomalous points per affected trace series), with currencyservice-related traces showing the largest relative latency surges. Earliest clustered anomalies occur around 17:36 UTC and additional spikes around ~17:52–17:53 UTC for some flows.\n\n"
    "Actionable next steps: investigate currencyservice and its interactions (especially incoming root/from_root and calls from checkoutservice) around 17:36 and ~17:52 UTC; inspect traces/backtraces for those timestamps to find the problematic span(s) and check host-level causes (CPU, network, socket) and downstream service health."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                      trace_name       p95  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp  max_relative_violation
21        currencyservice                    trace.from_root.duration_p95  0.000260                      25                        2  2024-01-15 17:36:00+00:00              149.096635
20        currencyservice                   trace.from_root.duration_mean  0.000087                      25                        2  2024-01-15 17:36:00+00:00               82.175392
9         checkoutservice           trace.to_currencyservice.duration_p95  0.000291                      25                        2  2024-01-15 17:52:00+00:00               48.906195
17        currencyservice         trace.from_checkoutservice.duration_p95  0.000291                      25                        2  2024-01-15 17:52:00+00:00               48.906195
13        checkoutservice            trace.to_paymentservice.duration_p95  0.001509                      25                        2  2024-01-15 17:31:00+00:00                0.941942
30  productcatalogservice  trace.from_recommendationservice.duration_mean  0.000242                      25                        2  2024-01-15 17:44:00+00:00                0.851400
40  recommendationservice    trace.to_productcatalogservice.duration_mean  0.000242                      25                        2  2024-01-15 17:44:00+00:00                0.851400
32  productcatalogservice                   trace.from_root.duration_mean  0.026391                      25                        2  2024-01-15 17:50:00+00:00                0.643931
18        currencyservice        trace.from_frontendservice.duration_mean  0.001273                      25                        2  2024-01-15 17:40:00+00:00                0.411651
2         checkoutservice        trace.from_frontendservice.duration_mean  0.468707                      25                        2  2024-01-15 17:30:00+00:00                0.406325
5         checkoutservice                    trace.from_root.duration_p95  0.149314                      25                        2  2024-01-15 17:39:00+00:00                0.236761
8         checkoutservice          trace.to_currencyservice.duration_mean  0.003343                      25                        2  2024-01-15 17:52:00+00:00                0.217436
16        currencyservice        trace.from_checkoutservice.duration_mean  0.003343                      25                        2  2024-01-15 17:52:00+00:00                0.217436
28  productcatalogservice        trace.from_frontendservice.duration_mean  0.000172                      25                        2  2024-01-15 17:47:00+00:00                0.203595
35  recommendationservice         trace.from_frontendservice.duration_p95  0.054340                      25                        2  2024-01-15 17:47:00+00:00                0.191456
11        checkoutservice              trace.to_emailservice.duration_p95  0.000567                      25                        2  2024-01-15 17:40:00+00:00                0.158713
23           emailservice         trace.from_checkoutservice.duration_p95  0.000567                      25                        2  2024-01-15 17:40:00+00:00                0.158713
14        checkoutservice    trace.to_productcatalogservice.duration_mean  0.000575                      25                        2  2024-01-15 17:46:00+00:00                0.144404
26  productcatalogservice        trace.from_checkoutservice.duration_mean  0.000575                      25                        2  2024-01-15 17:46:00+00:00                0.144404
0         checkoutservice        trace.from_checkoutservice.duration_mean  0.014728                      25                        2  2024-01-15 17:48:00+00:00                0.110051,                   timestamp          cmdb_id                             trace_name     value
0 2024-01-15 17:36:00+00:00  currencyservice           trace.from_root.duration_p95  0.038986
1 2024-01-15 17:52:00+00:00  currencyservice           trace.from_root.duration_p95  0.000294
2 2024-01-15 17:36:00+00:00  currencyservice          trace.from_root.duration_mean  0.007256
3 2024-01-15 17:52:00+00:00  currencyservice          trace.from_root.duration_mean  0.000091
4 2024-01-15 17:52:00+00:00  checkoutservice  trace.to_currencyservice.duration_p95  0.000299
5 2024-01-15 17:53:00+00:00  checkoutservice  trace.to_currencyservice.duration_p95  0.014524)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np

# Reuse existing metric if present; otherwise load and parse timestamp as UTC
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate focus: currencyservice
service = "currencyservice"

# Identify available KPI names for currencyservice (global)
currency_all = metric[metric['cmdb_id'] == service].copy()
kpis = sorted(currency_all['kpi_name'].dropna().unique())

# Compute global p95 and p5 and count_all per kpi_name using full series (do NOT filter by time)
thresholds = (
    currency_all
    .groupby('kpi_name')['value']
    .agg(count_all='count', p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# Incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-15 17:27:22", tz="UTC")
window_end   = pd.Timestamp("2024-01-15 17:57:22", tz="UTC")

# Filter currencyservice rows to the incident window
mask_win = (currency_all['timestamp'] >= window_start) & (currency_all['timestamp'] <= window_end)
currency_win = currency_all[mask_win].copy()

# Merge thresholds into windowed rows for per-row anomaly detection
currency_win = currency_win.merge(thresholds, on='kpi_name', how='left')

# Determine anomalies: high if value >= p95; low if value <= p5
currency_win['is_high'] = currency_win['value'] >= currency_win['p95']
currency_win['is_low']  = currency_win['value'] <= currency_win['p5']
currency_win['is_anom'] = currency_win['is_high'] | currency_win['is_low']

# Compute per-row relative violations:
# high: (value - p95)/p95 ; low: (p5 - value)/p5 ; if denom == 0 -> NaN
currency_win['rel_high'] = np.where(currency_win['p95'] != 0, (currency_win['value'] - currency_win['p95']) / currency_win['p95'], np.nan)
currency_win['rel_low']  = np.where(currency_win['p5']  != 0, (currency_win['p5'] - currency_win['value']) / currency_win['p5'], np.nan)

# Choose appropriate rel_violation per row and take absolute
currency_win['rel_violation'] = np.where(currency_win['is_high'], currency_win['rel_high'],
                                        np.where(currency_win['is_low'], currency_win['rel_low'], np.nan))
currency_win['rel_violation'] = currency_win['rel_violation'].abs()

# Aggregate required metrics per kpi_name
agg = (
    currency_win
    .groupby('kpi_name')
    .agg(
        total_points_in_window=('value', 'count'),
        anomaly_count_in_window=('is_anom', 'sum'),
        earliest_anomaly_timestamp=('timestamp', lambda x: x[currency_win.loc[x.index, 'is_anom']].min() if any(currency_win.loc[x.index, 'is_anom']) else pd.NaT),
        mean_value_in_window=('value', 'mean'),
        max_value_in_window=('value', 'max'),
        max_relative_violation_in_window=('rel_violation', 'max')
    )
    .reset_index()
)

# Combine with global thresholds and counts
result_df = thresholds.merge(agg, on='kpi_name', how='left')

# Fill NaNs for KPIs that had no window rows
result_df['total_points_in_window'] = result_df['total_points_in_window'].fillna(0).astype(int)
result_df['anomaly_count_in_window'] = result_df['anomaly_count_in_window'].fillna(0).astype(int)
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_timestamp']
result_df['mean_value_in_window'] = result_df['mean_value_in_window'].astype(float)
result_df['max_value_in_window'] = result_df['max_value_in_window'].astype(float)
result_df['max_relative_violation_in_window'] = result_df['max_relative_violation_in_window'].astype(float)

# Per requirement: if p95 or p5 == 0, relative violation should be NaN (already handled per-row). Ensure group-level compliance:
result_df.loc[result_df['p95'] == 0, 'max_relative_violation_in_window'] = np.nan
result_df.loc[result_df['p5'] == 0, 'max_relative_violation_in_window'] = result_df['max_relative_violation_in_window']  # keep as-is (only affects low anomalies)

# Define anomaly_count as points where value >= p95 OR value <= p5 (already anomaly_count_in_window)
# Sort by anomaly_count_in_window desc then max_relative_violation_in_window desc
result_df = result_df.sort_values(by=['anomaly_count_in_window', 'max_relative_violation_in_window'], ascending=[False, False])

# Keep requested columns and ensure compactness
result_df = result_df[['kpi_name', 'count_all', 'p95', 'p5',
                       'total_points_in_window', 'anomaly_count_in_window',
                       'earliest_anomaly_timestamp', 'mean_value_in_window',
                       'max_value_in_window', 'max_relative_violation_in_window']]

# Example metric rows for currencyservice from the window ordered by timestamp asc (up to 10)
examples_df = currency_win.sort_values('timestamp').loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value']].head(10).reset_index(drop=True)

# Display compact results: result_df (all currencyservice KPIs) and examples_df (<=10 rows)
result_df, examples_df

```
Out[9]:
```
Summary for currencyservice (window 2024-01-15 17:27:22 — 2024-01-15 17:57:22 UTC):

- KPIs available: cpu, latency-50, latency-90, mem, socket, workload.

- Global thresholds (full-series p95 / p5) and window anomaly highlights:
  - socket: count_all=25, p95=3.0, p5=3.0. Window points=25, anomalies=25 (earliest 2024-01-15 17:30:00 UTC). mean=3.0, max=3.0. (All window points meet the p95 threshold; relative violation = 0 because values equal p95.)
  - cpu: count_all=25, p95≈18.61, p5≈16.46. Window points=25, anomalies=4 (earliest 17:30 UTC). mean≈17.78, max≈19.28, max relative violation≈0.0577 (≈5.8% above p95).
  - workload: count_all=25, p95≈82.70, p5≈70.47. Window points=25, anomalies=4 (earliest 17:30 UTC). mean≈76.75, max≈85.37, max relative violation≈0.0363.
  - latency-50: count_all=25, p95≈0.06449, p5≈0.04044. Window points=25, anomalies=4 (earliest 17:31 UTC). mean≈0.05247, max≈0.06661, max relative violation≈0.0329.
  - latency-90: count_all=25, p95≈0.10409, p5≈0.09104. Window points=25, anomalies=4 (earliest 17:31 UTC). mean≈0.09536, max≈0.10633, max relative violation≈0.0216.
  - mem: count_all=25, p95≈42,119,050, p5≈41,416,700. Window points=25, anomalies=4 (earliest 17:34 UTC). mean≈41,777,770, max≈42,226,550, max relative violation≈0.0026.

- Interpretation:
  - The clearest signal is the socket KPI: every sampled point in the incident window met or exceeded the global p95 (25/25). This likely indicates a socket-related condition affecting currencyservice during the window (but because values equal the p95, it could be a thresholding artifact or a sustained shifted level).
  - CPU, workload, latency (50/90), and memory each show a small number of high anomalies (4 points each), concentrated near ~17:30–17:34 UTC, with modest relative deviations (few-percent level except socket which equals p95).
  - Example raw metric points around the critical times are available (first 10 from the window include CPU=15.51, latency-50≈0.04487, latency-90≈0.09386, mem≈4.211e7, socket=3.0, workload≈67.91 at 17:30 UTC; additional points at 17:31, etc.).

- Recommended next steps:
  - Investigate socket behavior for currencyservice (socket-level errors/connection counts) and verify whether the global p95=3.0 reflects normal baseline or a shifted baseline.
  - Inspect the ~17:30–17:36 UTC timeframe (and also ~17:52 from earlier trace findings) in traces/logs for spans related to currencyservice to pinpoint root cause (timeouts, downstream latency, or network/socket issues).
  - Correlate these metric anomalies with the trace-duration spikes already observed for currencyservice (notably trace.from_root.duration) to find concrete failing spans.

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name  count_all           p95            p5  total_points_in_window  anomaly_count_in_window earliest_anomaly_timestamp  mean_value_in_window  max_value_in_window  max_relative_violation_in_window
4      socket         25  3.000000e+00  3.000000e+00                      25                       25  2024-01-15 17:30:00+00:00          3.000000e+00         3.000000e+00                          0.000000
0         cpu         25  1.861362e+01  1.646174e+01                      25                        4  2024-01-15 17:30:00+00:00          1.777935e+01         1.928313e+01                          0.057681
5    workload         25  8.269847e+01  7.046819e+01                      25                        4  2024-01-15 17:30:00+00:00          7.674712e+01         8.537097e+01                          0.036343
1  latency-50         25  6.448899e-02  4.043762e-02                      25                        4  2024-01-15 17:31:00+00:00          5.247105e-02         6.661024e-02                          0.032893
2  latency-90         25  1.040887e-01  9.103790e-02                      25                        4  2024-01-15 17:31:00+00:00          9.536002e-02         1.063329e-01                          0.021561
3         mem         25  4.211905e+07  4.141670e+07                      25                        4  2024-01-15 17:34:00+00:00          4.177777e+07         4.222655e+07                          0.002552,                   timestamp          cmdb_id    kpi_name         value
0 2024-01-15 17:30:00+00:00  currencyservice         cpu  1.551221e+01
1 2024-01-15 17:30:00+00:00  currencyservice  latency-50  4.487284e-02
2 2024-01-15 17:30:00+00:00  currencyservice  latency-90  9.385683e-02
3 2024-01-15 17:30:00+00:00  currencyservice         mem  4.211367e+07
4 2024-01-15 17:30:00+00:00  currencyservice      socket  3.000000e+00
5 2024-01-15 17:30:00+00:00  currencyservice    workload  6.790716e+01
6 2024-01-15 17:31:00+00:00  currencyservice    workload  8.537097e+01
7 2024-01-15 17:31:00+00:00  currencyservice         mem  4.180582e+07
8 2024-01-15 17:31:00+00:00  currencyservice      socket  3.000000e+00
9 2024-01-15 17:31:00+00:00  currencyservice  latency-50  6.661024e-02)```
```